# RA-MedFiLM-BARR — Phase-wise ChatGPT Handoff Builder
## PACKAGING / NAVIGATION TOOL ONLY — NOT SCIENTIFIC AUTHORITY

Collects the reviewable source + governance + audit material for ONE
`(PHASE, REVISION)` profile into a single compact ZIP so that future
research review needs **one upload** instead of many individual files.

**This notebook performs no scientific execution.** It does not import
`torch`, load checkpoints, build models, run forward passes, execute smoke
or science, read VAL images / GT masks, read the TEST split, read
ROUTER_HOLDOUT, or train anything. It only copies and hashes existing
files, and writes a manifest + summary + ZIP under `reports/chatgpt_handoff/`.

**Checkpoint binaries (`.pt/.pth/.ckpt`) and datasets are NEVER copied into
the ZIP** — only their hashes/paths may appear inside the manifest/audits.

**A missing required artifact is recorded, never recreated.** Missing files
are listed in `HANDOFF_INDEX.json -> missing_files` and in
`HANDOFF_SUMMARY.md -> Missing files`.

### Phase-wise design
Profiles are declared explicitly in the `PROFILES` dict (Section 2).
Future profiles — e.g. `(37, "v1_0")`, `(38, "v1_0")`, `(39, "v1_0")` —
**must be added by hand**; the notebook never infers future scientific
files automatically.

## 1. Runtime / interpreter identity + imports
Confirms the canonical project interpreter and the allowed import set.

In [1]:
import sys, os, json, hashlib, zipfile, shutil, datetime
from pathlib import Path

EXPECTED_INTERPRETER = Path(r"D:\Softfors\jup\.venv\Scripts\python.exe")
print("sys.executable     :", sys.executable)
_ok = os.path.normcase(os.path.realpath(sys.executable)) == \
      os.path.normcase(os.path.realpath(str(EXPECTED_INTERPRETER)))
print("canonical venv match:", _ok)
assert _ok, ("WRONG INTERPRETER — run with " + str(EXPECTED_INTERPRETER))

# Forbidden-import guard (defence in depth; the static audit is authoritative).
for _mod in ("torch", "tensorflow", "cv2", "sklearn"):
    assert _mod not in sys.modules, f"forbidden module already imported: {_mod}"
print("allowed imports only  : OK")
print("STDLIB packaging ops   : read_bytes / sha256 / shutil.copy2 / zipfile")


sys.executable     : D:\Softfors\jup\.venv\Scripts\python.exe
canonical venv match: True
allowed imports only  : OK
STDLIB packaging ops   : read_bytes / sha256 / shutil.copy2 / zipfile


## 2. Phase-wise PROFILES — declare `(PHASE, REVISION)` explicitly

Each profile fixes:
* `root`-relative **required source files**,
* **required governance/audit files**,
* **minimal lineage/evidence files** (history),
* **expected raw-byte SHA-256 pins** where known (empty ⇒ not pinned yet).

> Current target: **Phase 36 / v1.14**. Add future phases here by hand.

In [2]:
ROOT = Path(r"D:\Softfors\jup")
assert (ROOT / "reports" / "audits").is_dir(), ROOT

PROFILES = {
    (36, "v1_14"): {
        "source_files": [
            "scripts/analysis/phase36_robustness_execution_v1_14.py",
            "scripts/analysis/launch_phase36_v1_14.py",
            "scripts/analysis/phase36_robustness_unit_v1_14.py",
            "scripts/analysis/phase36_robustness_preflight_v1_14.py",
            "scripts/analysis/phase36_robustness_negative_controls_v1_14.py",
        ],
        "audit_files": [
            "reports/audits/PHASE36_CURRENT_FILE_INDEX.md",
            "reports/audits/phase36_v1_14_runtime_source_integrity_manifest.json",
            "reports/audits/phase36_preexecution_source_freeze_v1_14.json",
            "reports/audits/phase36_synthetic_smoke_eligibility_v1_14.json",
            "reports/audits/phase36_v1_13_to_v1_14_semantic_diff.json",
            "reports/audits/phase36_v1_14_direct_source_byte_audit.json",
            "reports/audits/phase36_v1_14_no_scientific_execution_audit.json",
            "reports/audits/phase36_v1_14_science_release_attestation_contract.json",
            "reports/audits/phase36_v1_14_launcher_co_edit_negative_control.json",
            "reports/audits/phase36_robustness_unit_v1_14.json",
            "reports/audits/phase36_robustness_preflight_v1_14.json",
            "reports/audits/phase36_v1_14_negative_controls.json",
        ],
        "lineage_files": [
            "reports/audits/phase36_robustness_smoke_start_v1_12.json",
            "reports/audits/phase36_robustness_smoke_failure_v1_12.json",
            "reports/audits/phase36_v1_12_smoke_console_first_attempt.txt",
            "reports/audits/phase36_v1_12_cuda_environment_diagnosis.json",
            "reports/audits/phase36_v1_12_failed_smoke_adjudication.json",
            "reports/audits/phase36_preexecution_source_freeze_v1_13.json",
            "reports/audits/phase36_synthetic_smoke_eligibility_v1_13.json",
        ],
        # raw-byte SHA-256 pins (files verified byte-for-byte before packaging)
        "pins": {
            "scripts/analysis/phase36_robustness_execution_v1_14.py":
                "ac09ff8ded3610244920e66612145b246b33127140d344a9730d454bab27cff5",
            "scripts/analysis/launch_phase36_v1_14.py":
                "2339d6afa1507d68decd1b63b3f3250d86ecd01659580394c6f8dace4052e2c3",
            "scripts/analysis/phase36_robustness_unit_v1_14.py":
                "f837de2421ca464910b1b41c1ac29c5eae8fb686ba01e8b41fc499305b8ab7e7",
            "scripts/analysis/phase36_robustness_preflight_v1_14.py":
                "5ef73cc0a468c736ebce577c7be8186adbd37996ec48ddb29ed43e9c341143b8",
            "scripts/analysis/phase36_robustness_negative_controls_v1_14.py":
                "640b69b158018885a8990b9759f86aef9bf92097cbfe3dc48d57c334a9f6559a",
            "reports/audits/phase36_v1_14_runtime_source_integrity_manifest.json":
                "b4c358dfcb5e7b59beb6ae1ba2cffc64677d10049e12568b4c8eea8e79bb911e",
            "reports/audits/phase36_preexecution_source_freeze_v1_14.json":
                "332e6ad8db0d62baf19fcc10ce2809daee367b38cf56f7cda99a95f049e64655",
            "reports/audits/phase36_synthetic_smoke_eligibility_v1_14.json":
                "63d9fd04d86541629675669cda4c7ed0b5482ff51a22b08b110816e4cfccffbb",
            "reports/audits/phase36_v1_13_to_v1_14_semantic_diff.json":
                "45d382c6180b449587c1af301ab51163057b89f3933dc4278c5f0b969ff4c3f8",
        },
    },
    # ---- FUTURE PROFILES: add explicitly, e.g.
    # (37, "v1_0"): {"source_files": [...], "audit_files": [...],
    #               "lineage_files": [...], "pins": {...}},
}

PHASE = 36
REVISION = "v1_14"
PROFILE = PROFILES[(PHASE, REVISION)]
print("profile loaded:", (PHASE, REVISION))
print("  source :", len(PROFILE["source_files"]))
print("  audit  :", len(PROFILE["audit_files"]))
print("  lineage:", len(PROFILE["lineage_files"]))
print("  pins   :", len(PROFILE["pins"]))


profile loaded: (36, 'v1_14')
  source : 5
  audit  : 12
  lineage: 7
  pins   : 9


## 3. Raw-byte hashing helper (NEVER text-normalized)
`sha256` is always computed over `read_bytes()` so CRLF/LF is preserved.

In [3]:
def sha256_bytes(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# sanity: hashing is raw-byte and stable
_probe = ROOT / PROFILE["audit_files"][0]
print("probe:", _probe.name, sha256_bytes(_probe)[:16], "...")
print("raw-byte hashing: OK (no CRLF/LF normalization)")


probe: PHASE36_CURRENT_FILE_INDEX.md ca28cb8d3bc499de ...
raw-byte hashing: OK (no CRLF/LF normalization)


## 4. Verify pinned SHAs (raw-byte) BEFORE packaging
Each pinned path must exist and hash exactly. Any mismatch ⇒ STOP.

In [4]:
pin_results = {}
for rel, expected in PROFILE["pins"].items():
    p = ROOT / rel
    if not p.is_file():
        pin_results[rel] = {"present": False, "match": False,
                            "expected": expected, "observed": None}
        continue
    obs = sha256_bytes(p)
    pin_results[rel] = {"present": True, "match": (obs == expected),
                        "expected": expected, "observed": obs}

all_expected_pins_match = all(v["match"] for v in pin_results.values())
for rel, v in pin_results.items():
    print(("MATCH " if v["match"] else "FAIL  ") + rel)
print("all_expected_pins_match:", all_expected_pins_match)
assert all_expected_pins_match, "PIN MISMATCH — STOP; do not package."


MATCH scripts/analysis/phase36_robustness_execution_v1_14.py
MATCH scripts/analysis/launch_phase36_v1_14.py
MATCH scripts/analysis/phase36_robustness_unit_v1_14.py
MATCH scripts/analysis/phase36_robustness_preflight_v1_14.py
MATCH scripts/analysis/phase36_robustness_negative_controls_v1_14.py
MATCH reports/audits/phase36_v1_14_runtime_source_integrity_manifest.json
MATCH reports/audits/phase36_preexecution_source_freeze_v1_14.json
MATCH reports/audits/phase36_synthetic_smoke_eligibility_v1_14.json
MATCH reports/audits/phase36_v1_13_to_v1_14_semantic_diff.json
all_expected_pins_match: True


## 5. Resolve the handoff file set (missing files recorded, never recreated)

In [5]:
def _classify(rels, kind):
    present, missing = [], []
    for rel in rels:
        (present if (ROOT / rel).is_file() else missing).append(rel)
    return {"kind": kind, "present": present, "missing": missing}

groups = [
    _classify(PROFILE["source_files"], "source"),
    _classify(PROFILE["audit_files"], "audit"),
    _classify(PROFILE["lineage_files"], "lineage"),
]
present_files = [rel for g in groups for rel in g["present"]]
missing_files = sorted(rel for g in groups for rel in g["missing"])

print("present:", len(present_files), " missing:", len(missing_files))
if missing_files:
    print("MISSING (recorded, NOT recreated):")
    for m in missing_files:
        print("  -", m)


present: 24  missing: 0


## 6. Build the bundle directory tree
`reports/chatgpt_handoff/phase<PHASE>/<REVISION>_<UTC>/`

In [6]:
utc = datetime.datetime.now(datetime.timezone.utc)
STAMP = utc.strftime("%Y%m%dT%H%M%SZ")
BASE = ROOT / "reports" / "chatgpt_handoff" / f"phase{PHASE}"
BUNDLE = BASE / f"{REVISION}_{STAMP}"
BUNDLE.mkdir(parents=True, exist_ok=True)
(BUNDLE / "scripts").mkdir(exist_ok=True)
(BUNDLE / "reports").mkdir(exist_ok=True)
print("bundle dir:", BUNDLE)


bundle dir:

 D:\Softfors\jup\reports\chatgpt_handoff\phase36\v1_14_20261006T171552Z


## 7. Copy allowed material (raw bytes via `shutil.copy2`)
Source → `scripts/…`, audits/lineage → `reports/audits/…`.
**No `.pt/.pth/.ckpt`, no datasets, no TEST/ROUTER_HOLDOUT.**

In [7]:
FORBIDDEN_SUFFIXES = (".pt", ".pth", ".ckpt", ".npy", ".npz", ".pkl")

copied = []
for rel in present_files:
    src = ROOT / rel
    if src.suffix.lower() in FORBIDDEN_SUFFIXES:
        raise SystemExit(f"REFUSE to package binary/checkpoint: {rel}")
    dst = BUNDLE / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copied.append({"rel": rel, "sha256": sha256_bytes(src),
                   "size_bytes": src.stat().st_size})

print("copied files:", len(copied))
assert len(copied) == len(present_files)


copied files: 24


## 8. Write `HANDOFF_INDEX.json` (machine-readable manifest)

In [8]:
by_kind = {}
for rel in present_files:
    if rel.startswith("scripts/"):
        k = "source"
    elif rel in PROFILE["lineage_files"]:
        k = "lineage"
    else:
        k = "audit"
    by_kind.setdefault(k, []).append(rel)

handoff_index = {
    "artifact": "phase36_chatgpt_handoff_index",
    "phase": PHASE,
    "revision": REVISION,
    "revision_id": "ra_medfilm_barr_phase36_robustness_v1_14",
    "generated_utc": utc.isoformat(),
    "purpose": "PHASE_WISE_CHATGPT_HANDOFF_PACKAGING_ONLY",
    "not_scientific_authority": True,
    "root": str(ROOT),
    "bundle_directory": str(BUNDLE),
    "interpreter": sys.executable,
    "n_files": len(copied),
    "files": copied,
    "by_kind": by_kind,
    "missing_files": missing_files,
    "pins": pin_results,
    "all_expected_pins_match": all_expected_pins_match,
    "checkpoint_binaries_included": False,
    "dataset_files_included": False,
    "test_files_included": False,
    "router_holdout_files_included": False,
    "smoke_run": False,
    "science_run": False,
    "model_execution": False,
    "scientific_output": False,
}
(BUNDLE / "HANDOFF_INDEX.json").write_text(
    json.dumps(handoff_index, indent=1), encoding="utf-8")
print("wrote:", BUNDLE / "HANDOFF_INDEX.json")


wrote: D:\Softfors\jup\reports\chatgpt_handoff\phase36\v1_14_20261006T171552Z\HANDOFF_INDEX.json


## 9. Write `HANDOFF_SUMMARY.md` (human-readable)

In [9]:
lines = []
lines.append(f"# Phase {PHASE} {REVISION} — ChatGPT Handoff Summary")
lines.append("")
lines.append("**PACKAGING / NAVIGATION ONLY — NOT scientific authority.**")
lines.append("")
lines.append(f"* Generated (UTC): `{utc.isoformat()}`")
lines.append(f"* Interpreter: `{sys.executable}`")
lines.append(f"* Files packaged: **{len(copied)}**")
lines.append(f"* All expected pins match: **{all_expected_pins_match}**")
lines.append("")
lines.append("## Evidence-class scope")
lines.append("")
lines.append("No smoke, no science, no model/checkpoint, no VAL/GT, "
             "no TEST, no ROUTER_HOLDOUT, no training was performed.")
lines.append("Checkpoint binaries and datasets are NOT included.")
lines.append("")
lines.append("## Packaged files")
lines.append("")
for k in ("source", "audit", "lineage"):
    rels = by_kind.get(k, [])
    if not rels:
        continue
    lines.append(f"### {k} ({len(rels)})")
    lines.append("")
    for rel in rels:
        lines.append(f"* `{rel}`")
lines.append("")
lines.append("## Missing files")
lines.append("")
if missing_files:
    for m in missing_files:
        lines.append(f"* `{m}`")
else:
    lines.append("_None — all required files were present._")
lines.append("")
lines.append("## Pinned SHA-256 (raw-byte)")
lines.append("")
for rel, v in pin_results.items():
    lines.append(f"* `{rel}`")
    lines.append(f"  * expected `{v['expected']}`")
    lines.append(f"  * observed `{v['observed']}`")
    lines.append(f"  * match **{v['match']}**")
lines.append("")
lines.append("## Current research state (UNCHANGED by this task)")
lines.append("")
lines.append("* v1.12 = CONSUMED_FAILED_PRE_MODEL / RETRY_FORBIDDEN")
lines.append("* v1.13 = FROZEN_NEVER_CONSUMED")
lines.append("* v1.14 smoke = ELIGIBLE_NOT_STARTED")
lines.append("* v1.14 science = INELIGIBLE_NOT_STARTED")
lines.append("* Phase37 = NOT_STARTED")
lines.append("* TEST = REMAINS_PHASE39_ONLY")
(BUNDLE / "HANDOFF_SUMMARY.md").write_text(
    "\n".join(lines) + "\n", encoding="utf-8")
print("wrote:", BUNDLE / "HANDOFF_SUMMARY.md")


wrote:

 D:\Softfors\jup\reports\chatgpt_handoff\phase36\v1_14_20261006T171552Z\HANDOFF_SUMMARY.md


## 10. Build the ZIP and inspect its members
Fails if any forbidden member appears (checkpoints / datasets / TEST / ROUTER_HOLDOUT).

In [10]:
ZIP_PATH = BASE / f"{REVISION}_{STAMP}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(BUNDLE.rglob("*")):
        if f.is_file():
            zf.write(f, arcname=str(f.relative_to(BASE)))

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    members = zf.namelist()

_forbidden_suffixes = (".pt", ".pth", ".ckpt", ".npy", ".npz", ".pkl")
_forbidden_tokens = ("test_set", "router_holdout", "ground_truth", "gt_mask",
                     "images/", "masks/", "dataset", "/data/")
bad = []
for m in members:
    low = m.lower()
    if low.endswith(_forbidden_suffixes) or any(t in low for t in _forbidden_tokens):
        bad.append(m)
print("zip members:", len(members))
print("forbidden members:", bad)
assert not bad, f"FORBIDDEN ZIP MEMBERS: {bad}"
for m in members:
    print("  ", m)


zip members: 26
forbidden members: []
   v1_14_20261006T171552Z/HANDOFF_INDEX.json
   v1_14_20261006T171552Z/HANDOFF_SUMMARY.md
   v1_14_20261006T171552Z/reports/audits/PHASE36_CURRENT_FILE_INDEX.md
   v1_14_20261006T171552Z/reports/audits/phase36_preexecution_source_freeze_v1_13.json
   v1_14_20261006T171552Z/reports/audits/phase36_preexecution_source_freeze_v1_14.json
   v1_14_20261006T171552Z/reports/audits/phase36_robustness_preflight_v1_14.json
   v1_14_20261006T171552Z/reports/audits/phase36_robustness_smoke_failure_v1_12.json
   v1_14_20261006T171552Z/reports/audits/phase36_robustness_smoke_start_v1_12.json
   v1_14_20261006T171552Z/reports/audits/phase36_robustness_unit_v1_14.json
   v1_14_20261006T171552Z/reports/audits/phase36_synthetic_smoke_eligibility_v1_13.json
   v1_14_20261006T171552Z/reports/audits/phase36_synthetic_smoke_eligibility_v1_14.json
   v1_14_20261006T171552Z/reports/audits/phase36_v1_12_cuda_environment_diagnosis.json
   v1_14_20261006T171552Z/reports/audit

## 11. ZIP integrity summary (sha-256 / size / count)

In [11]:
zip_sha = sha256_bytes(ZIP_PATH)
zip_size = ZIP_PATH.stat().st_size
print("zip           :", ZIP_PATH)
print("zip sha256    :", zip_sha)
print("zip size bytes:", zip_size)
print("members       :", len(members))
print("missing files :", len(missing_files))


zip           : D:\Softfors\jup\reports\chatgpt_handoff\phase36\v1_14_20261006T171552Z.zip
zip sha256    : 1af88d843859a1d9aa4e8e489f18f32a048749db31f8de62ec22cb65e2c5bb57
zip size bytes: 116467
members       : 26
missing files : 0


## 12. Write the generation audit
`reports/audits/phase36_v1_14_chatgpt_handoff_generation.json`
(path derived from PHASE/REVISION so future profiles write their own).

In [12]:
AUD = ROOT / "reports" / "audits"
AUD.mkdir(parents=True, exist_ok=True)
gen_audit_name = f"phase{PHASE}_{REVISION}_chatgpt_handoff_generation.json"
GEN_AUDIT = AUD / gen_audit_name
generation = {
    "artifact": f"phase{PHASE}_{REVISION}_chatgpt_handoff_generation",
    "phase": PHASE,
    "revision": REVISION,
    "generated_utc": utc.isoformat(),
    "interpreter": sys.executable,
    "bundle_directory": str(BUNDLE),
    "zip_path": str(ZIP_PATH),
    "zip_sha256": zip_sha,
    "zip_size_bytes": zip_size,
    "number_of_files": len(copied),
    "zip_member_count": len(members),
    "missing_files": missing_files,
    "all_expected_pins_match": all_expected_pins_match,
    "checkpoint_binaries_included": False,
    "dataset_files_included": False,
    "test_files_included": False,
    "router_holdout_files_included": False,
    "smoke_run": False,
    "science_run": False,
    "model_execution": False,
    "scientific_output": False,
    "terminal": f"PHASE{PHASE}_{REVISION.upper()}_CHATGPT_HANDOFF_PACKAGE_COMPLETE",
}
GEN_AUDIT.write_text(json.dumps(generation, indent=1), encoding="utf-8")
print("wrote:", GEN_AUDIT)
print("terminal:", generation["terminal"])
print("NEXT: upload the ZIP to ChatGPT for the independent source-level audit.")


wrote: D:\Softfors\jup\reports\audits\phase36_v1_14_chatgpt_handoff_generation.json
terminal: PHASE36_V1_14_CHATGPT_HANDOFF_PACKAGE_COMPLETE
NEXT: upload the ZIP to ChatGPT for the independent source-level audit.
